## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1

В данной части работы рассмотрены:
* загрузка данных из HDFS;
* базовые преобразования данных;
* загрузка преобразованных данных в таблицу `Apache Iceberg`.

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    col,
    trim,
    unix_timestamp
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    user_name = os.getenv("USER")
    
    conf = SparkConf()
    conf.setAppName("lab 1 US Accidents")
    conf.setMaster("yarn")
    conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "8")
    conf.set("spark.executor.instances", "2")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "2")
    conf.set("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.0")
    conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    conf.set("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkCatalog")
    conf.set("spark.sql.catalog.spark_catalog.type", "hadoop")
    conf.set("spark.sql.catalog.spark_catalog.warehouse", f"hdfs:///user/{user_name}/warehouse")
    conf.set("spark.sql.catalog.spark_catalog.io-impl", "org.apache.iceberg.hadoop.HadoopFileIO")

    return conf

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`. В процессе создания сессии происходит подключение к кластеру `Apache Hadoop`, что может занять некоторое время.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

:: loading settings :: url = jar:file:/opt/spark-3.5.2-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/user7/.ivy2/cache
The jars for the packages stored in: /home/user7/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-814e8178-91b1-41f7-bbdf-8194c6329b8d;1.0
	confs: [default]


	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 in central
:: resolution report :: resolve 571ms :: artifacts dl 21ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-814e8178-91b1-41f7-bbdf-8194c6329b8d
	confs: [default]


	0 artifacts copied, 1 already retrieved (0kB/26ms)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/12 17:00:45 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


26/10/12 17:01:03 WARN Client: Same path resource file:///home/user7/.ivy2/jars/org.apache.iceberg_iceberg-spark-runtime-3.5_2.12-1.6.0.jar added multiple times to distributed cache.


Для исследования будем использовать датасет `"US Accidents (2016 - 2023)"`, расположенный на платформе `Kaggle` по адресу https://www.kaggle.com/datasets/sobhanmoosavi/us-accidents.

Датасет содержит 7 728 394 записи о дорожно-транспортных происшествиях, случившихся в 49 штатах США с февраля 2016 по март 2023 года. Данные собирались службой `AccidentPrevention` через потоковые API транспорта, правоохранительных органов и дорожных датчиков. Каждый случай описан временем и местом происшествия, метеоусловиями и признаками места события (переезд, перекрёсток, остановка и т.д.), а также степенью тяжести (`Severity`).

Исходный файл весит около 2.9 ГБ и представляет собой один CSV-файл с 46 столбцами, поэтому его обработка выполняется средствами `Apache Spark`.

Данный датасет загружен в `HDFS` по адресу: `hdfs:///user/user7/otpushchennikov/data/US_Accidents_March23.csv`

Указываем путь в `HDFS` для файла с данными.

In [5]:
path = "hdfs:///user/user7/otpushchennikov/data/US_Accidents_March23.csv"

Заполняем датафрейм данными из файла.

Здесь важно не добавить напрасно опцию `multiLine`, которую часто рекомендуют для `CSV`. Она разрешает одной записи занимать несколько физических строк файла (так происходит, если текстовое поле содержит перевод строки и заключено в кавычки), но из-за неё `Apache Spark` не может разбивать файл на части при чтении: весь файл целиком обрабатывается одной задачей, и для файла весом около 2.9 ГБ это приводит к часам ожидания.

Проверим, нужна ли такая опция нашим данным. В файле 7 728 395 строк, из них 7 728 394 начинаются с идентификатора записи вида `A-<номер>` (осталась только строка заголовка), то есть каждой записи соответствует ровно одна строка. Кроме того, в файле нет ни одного символа двойной кавычки, а каждая строка содержит ровно 45 разделителей, то есть 46 полей. Следовательно, `multiLine` не требуется, и файл будет читаться параллельно -- по разделению на каждый из 23 блоков `HDFS`, на которые он физически разбит.

In [6]:
df = (spark.read.format("csv")
      .option("header", "true")
      .load(path)
)

Выводим фрагмент датафрейма на экран.

In [7]:
df.show(3, truncate=25)

26/10/12 17:02:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+-------+--------+-------------------+-------------------+-----------------+----------+-------+-------+------------+-------------------------+--------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
| ID| Source|Severity|         Start_Time|           End_Time|        Start_Lat| Start_Lng|End_Lat|End_Lng|Distance(mi)|              Description|        Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|Amenity| Bump|Crossing|

Очевидно, что в целях сохранения ясности изложения и сокращения расчетного времени имеет смысл рассматривать не все столбцы датасета. Оставим следующие колонки, удалив остальные:

| Название столбца  | Расшифровка |
| ------------- | ------------- |
| ID | Идентификатор записи о происшествии |
| Severity | Степень тяжести происшествия (от 1 до 4) |
| Start_Time | Дата и время начала происшествия |
| End_Time | Дата и время окончания происшествия |
| Start_Lat | Широта места начала происшествия |
| Start_Lng | Долгота места начала происшествия |
| Distance(mi) | Протяжённость перекрытого участка дороги, мили |
| Street | Улица, на которой произошло происшествие |
| City | Населённый пункт |
| County | Округ |
| State | Штат |
| Zipcode | Почтовый индекс |
| Timezone | Часовой пояс |
| Airport_Code | Код ближайшего аэропорта |
| Weather_Timestamp | Дата и время метеосводки |
| Temperature(F) | Температура воздуха, фаренгейты |
| Wind_Chill(F) | Ощущаемая температура с учётом ветра, фаренгейты |
| Humidity(%) | Относительная влажность воздуха |
| Pressure(in) | Атмосферное давление, дюймы ртутного столба |
| Visibility(mi) | Видимость, мили |
| Wind_Direction | Направление ветра |
| Wind_Speed(mph) | Скорость ветра, мили в час |
| Precipitation(in) | Количество осадков, дюймы |
| Weather_Condition | Метеоусловия (дождь, снег, туман и т.д.) |
| Bump | Наличие неровности (лежачего полицейского) на участке |
| Crossing | Наличие пересечения проезжих частей |
| Give_Way | Наличие знака «уступите дорогу» |
| Junction | Наличие стыковки/примыкания дорог |
| No_Exit | Отсутствие съезда с участка |
| Railway | Наличие железнодорожного переезда |
| Roundabout | Наличие кругового движения |
| Station | Наличие ближайшей станции |
| Stop | Наличие остановки |
| Traffic_Signal | Наличие светофора |
| Turning_Loop | Наличие петли для разворота |
| Sunrise_Sunset | Светлое или тёмное время суток |
| Civil_Twilight | Гражданские сумерки |
| Nautical_Twilight | Навигационные сумерки |
| Astronomical_Twilight | Астрономические сумерки |

In [8]:
df = df.select(
    "ID", "Severity", "Start_Time", "End_Time", "Start_Lat", "Start_Lng", "Distance(mi)",
    "Street", "City", "County", "State", "Zipcode", "Timezone", "Airport_Code",
    "Weather_Timestamp", "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Direction", "Wind_Speed(mph)", "Precipitation(in)",
    "Weather_Condition", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway",
    "Roundabout", "Station", "Stop", "Traffic_Signal", "Turning_Loop", "Sunrise_Sunset",
    "Civil_Twilight", "Nautical_Twilight", "Astronomical_Twilight"
)

In [9]:
df.show(3, truncate=25)

+---+--------+-------------------+-------------------+-----------------+----------+------------+--------------+------------+----------+-----+----------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-----+--------+--------+--------+-------+-------+----------+-------+-----+--------------+------------+--------------+--------------+-----------------+---------------------+
| ID|Severity|         Start_Time|           End_Time|        Start_Lat| Start_Lng|Distance(mi)|        Street|        City|    County|State|   Zipcode|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition| Bump|Crossing|Give_Way|Junction|No_Exit|Railway|Roundabout|Station| Stop|Traffic_Signal|Turning_Loop|Sunrise_Sunset|Civil_Twilight|Nautical_Twilight|Astronomical_

Выведем на экран метаданные датасета.

In [10]:
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Severity: string (nullable = true)
 |-- Start_Time: string (nullable = true)
 |-- End_Time: string (nullable = true)
 |-- Start_Lat: string (nullable = true)
 |-- Start_Lng: string (nullable = true)
 |-- Distance(mi): string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: string (nullable = true)
 |-- Temperature(F): string (nullable = true)
 |-- Wind_Chill(F): string (nullable = true)
 |-- Humidity(%): string (nullable = true)
 |-- Pressure(in): string (nullable = true)
 |-- Visibility(mi): string (nullable = true)
 |-- Wind_Direction: string (nullable = true)
 |-- Wind_Speed(mph): string (nullable = true)
 |-- Precipitation(in): string (nullable = true)
 |-- Weather_Condition: st

Видно, что все столбцы датасета содержат строковый тип данных, что не соответствует ожиданиям. Выполним преобразования типов данных некоторых столбцов.

Дополнительно сформируем производный признак `Duration_min` -- длительность происшествия в минутах как разность `End_Time` и `Start_Time`. Он понадобится во второй части работы как цель регрессии, а в первой -- заодно позволит проверить корректность временны́х меток.

Значения пропусков и выбросов на этом этапе не обрабатываются: все такие проверки требуют полных сканов данных и выполняются во второй части работы уже по сохранённой таблице.

In [11]:
BOOLEAN_COLUMNS = [
    "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", "Roundabout",
    "Station", "Stop", "Traffic_Signal", "Turning_Loop"
]

FLOAT_COLUMNS = [
    "Distance(mi)", "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)",
    "Visibility(mi)", "Wind_Speed(mph)", "Precipitation(in)"
]

def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Преобразует столбцы DataFrame в указанные типы данных и
    выполняет необходимые преобразования.

    Args:
        data (DataFrame): Исходный DataFrame.

    Returns:
        DataFrame: Преобразованный DataFrame.
    """
    # Целевой признак и координаты
    data = data.withColumn("Severity",
                           col("Severity").cast("Integer"))
    data = data.withColumn("Start_Lat",
                           col("Start_Lat").cast("Double"))
    data = data.withColumn("Start_Lng",
                           col("Start_Lng").cast("Double"))

    # Временные метки
    data = data.withColumn("Start_Time",
                           col("Start_Time").cast("Timestamp"))
    data = data.withColumn("End_Time",
                           col("End_Time").cast("Timestamp"))
    data = data.withColumn("Weather_Timestamp",
                           col("Weather_Timestamp").cast("Timestamp"))

    # Производный признак: длительность происшествия в минутах
    data = data.withColumn("Duration_min",
                           ((unix_timestamp(col("End_Time")) - unix_timestamp(col("Start_Time"))) / 60).cast("Float"))

    # Числовые метеопризнаки: удалять единицы измерения не требуется,
    # они вынесены в заголовки столбцов
    for name in FLOAT_COLUMNS:
        data = data.withColumn(name, col(name).cast("Float"))

    # Булевы признаки инфраструктуры: в исходных данных это строки
    # 'True'/'False'
    for name in BOOLEAN_COLUMNS:
        data = data.withColumn(name, col(name).cast("Boolean"))

    # Идентификатор, почтовый индекс и адресные поля оставляем строками:
    # индекс -- не число, а код (ведущие нули значимы)
    data = data.withColumn("Zipcode",
                           trim(col("Zipcode")))

    return data

In [12]:
df = transform_dataframe(df)

In [13]:
df.show(3, truncate=25)

+---+--------+-------------------+-------------------+-----------------+----------+------------+--------------+------------+----------+-----+----------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-----+--------+--------+--------+-------+-------+----------+-------+-----+--------------+------------+--------------+--------------+-----------------+---------------------+------------+
| ID|Severity|         Start_Time|           End_Time|        Start_Lat| Start_Lng|Distance(mi)|        Street|        City|    County|State|   Zipcode|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition| Bump|Crossing|Give_Way|Junction|No_Exit|Railway|Roundabout|Station| Stop|Traffic_Signal|Turning_Loop|Sunrise_Sunset|Civil_Twilight|Nautical_Twilight|

In [14]:
df.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- Distance(mi): float (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): float (nullable = true)
 |-- Wind_Chill(F): float (nullable = true)
 |-- Humidity(%): float (nullable = true)
 |-- Pressure(in): float (nullable = true)
 |-- Visibility(mi): float (nullable = true)
 |-- Wind_Direction: string (nullable = true)
 |-- Wind_Speed(mph): float (nullable = true)
 |-- Precipitation(in): float (nullable = true)
 |-- Weather_Condition: 

Перед записью проверим, что данные действительно занимают несколько разделений (`partitions`) и запись в таблицу пойдёт параллельно всеми исполнителями, а не одной задачей. Вычислим число разделений датафрейма.

In [15]:
df.rdd.getNumPartitions()

23

Разделений 23 -- ровно по числу блоков `HDFS`, на которые разбит исходный файл, значит каждая часть будет записана своей задачей одновременно.

Видно, что теперь столбцы датафрейма содержат значения корректных типов.

Полученный датафрейм сохраним для дальнейшего использования. Сохранение выполним в таблицу `Apache Iceberg`.

Сначала создадим базу данных, в которой будет расположена таблица. Во избежание путаницы, **каждую таблицу следует называть с использованием имени пользователя кластера**.

In [16]:
database_name = "otpushchennikov_database"
table_name = "otpushchennikov_us_accidents"

Создадим инструкцию SQL для добавления базы данных в каталог `Apache Spark`.

In [17]:
create_database_sql = f"""
CREATE DATABASE IF NOT EXISTS spark_catalog.{database_name}
"""

In [18]:
spark.sql(create_database_sql)

DataFrame[]

Установим созданную базу данных как текущую.

In [19]:
spark.catalog.setCurrentDatabase(database_name)

И, наконец, записываем преобразованный датафрейм в таблицу `otpushchennikov_us_accidents`. Задача создания файлов таблицы поставится на все 23 разделения одновременно, поэтому запись занимает несколько минут.

In [20]:
# Сохранение DataFrame в виде таблицы
df.writeTo(table_name).using("iceberg").create()

После успешной записи можно посмотреть, какие таблицы входят в базу данных.

In [21]:
for table in spark.catalog.listTables():
    print(table.name)

otpushchennikov_us_accidents


Обратите внимание, что при необходимости созданные базу данных и таблицу можно удалить следующими командами.

In [22]:
# spark.sql("DROP TABLE spark_catalog.otpushchennikov_database.otpushchennikov_us_accidents")
# spark.sql("DROP DATABASE spark_catalog.otpushchennikov_database")

После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [23]:
spark.stop()